In [11]:
import pandas as pd
import csv

In [12]:
import pandas as pd
from pathlib import Path
import re

pasta_principal = Path(r'/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet')

padrao_nome = r'Nome:\s*(.*)' # Seu regex
padrao_codigo = r'Codigo Estacao:\s*(.*)' # Seu regex
padrao_latitude = r'Latitude:\s*(.*)' # Seu regex
padrao_longitude= r'Longitude:\s*(.*)' # Seu regex
padrao_altitude = r'Altitude:\s*(.*)' # Seu regex
padrao_situacao= r'Situacao:\s*(.*)' # Seu regex
colunas = [
    "Data Medicao", "EVAPORACAO DO PICHE, DIARIA(mm)", "INSOLACAO TOTAL, DIARIO(h)",
    "PRECIPITACAO TOTAL, DIARIO(mm)", "TEMPERATURA MAXIMA, DIARIA(°C)", 
    "TEMPERATURA MEDIA COMPENSADA, DIARIA(°C)", "TEMPERATURA MINIMA, DIARIA(°C)", 
    "UMIDADE RELATIVA DO AR, MEDIA DIARIA(%)", "UMIDADE RELATIVA DO AR, MINIMA DIARIA(%)", 
    "VENTO, VELOCIDADE MEDIA DIARIA(m/s)", "nome_estacao", "codigo_estacao", 
    "latitude_estacao", "longitude_estacao", "altitude_estacao", "situacao_estacao"
]
dados_final = pd.DataFrame(columns=colunas)
print(f"Iniciando busca por '*.csv' em: {pasta_principal}\n")

for arquivo in pasta_principal.rglob('*.csv'):
    try:
        print(f"--- Lendo {arquivo.name} ---")
        arquivo_final = Path(rf'/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet/{arquivo.name}_edit.csv')
        conteudo = arquivo.read_text(encoding='utf-8')
        nome_estacao = re.findall(padrao_nome, conteudo, re.DOTALL)
        codigo_estacao = re.findall(padrao_codigo, conteudo, re.DOTALL)
        latitude_estacao = re.findall(padrao_latitude, conteudo, re.DOTALL)
        longitude_estacao = re.findall(padrao_longitude, conteudo, re.DOTALL)
        altitude_estacao = re.findall(padrao_altitude, conteudo, re.DOTALL)
        situacao_estacao = re.findall(padrao_situacao, conteudo, re.DOTALL)


        with open(arquivo, 'r', encoding='utf-8') as f_entrada:
            leitor = csv.reader(f_entrada)
            linhas = list(leitor)

        linhas_filtradas = [linhas[0]] + linhas[11:]

        with open(arquivo_final, 'w', encoding='utf-8', newline='') as f_saida:
            escritor = csv.writer(f_saida)
            escritor.writerows(linhas_filtradas)

        dados = pd.read_csv(arquivo_final, sep=';', decimal='.', encoding='latin1')
        
        dados['nome_estacao'] = nome_estacao
        dados['codigo_estacao'] = codigo_estacao
        dados['latitude_estacao'] = latitude_estacao
        dados['longitude_estacao'] = longitude_estacao
        dados['altitude_estacao'] = altitude_estacao
        dados['situacao_estacao'] = situacao_estacao
            
        dados_final = pd.concat([dados, dados_final], ignore_index=True)
    except Exception as e:
        print(f"    Erro ao ler o arquivo {arquivo.name}: {e}")
dados_final.to_csv('/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet/dados.csv')

Iniciando busca por '*.csv' em: /home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet

--- Lendo dados_83481_D_2009-01-01_2018-01-31.csv ---
    Erro ao ler o arquivo dados_83481_D_2009-01-01_2018-01-31.csv: Length of values (1) does not match length of index (3318)
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv ---
    Erro ao ler o arquivo dados_83437_D_2009-01-01_2026-04-30.csv: Length of values (1) does not match length of index (6329)
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv_edit.csv ---
    Erro ao ler o arquivo dados_83437_D_2009-01-01_2026-04-30.csv_edit.csv: Length of values (1) does not match length of index (6319)
--- Lendo dados_83692_D_2009-01-01_2026-04-30.csv ---
    Erro ao ler o arquivo dados_83692_D_2009-01-01_2026-04-30.csv: Length of values (1) does not match length of index (6329)
--- Lendo dados_83587_D_2009-01-01_2026-04-30.csv ---
    Erro ao ler o arquivo dados_83587_D_2009-01-01_2026-04-30.csv: Length of values (1) does not match length of index (

In [17]:
import pandas as pd
from pathlib import Path
import re

pasta_principal = Path(r'/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet')

# Regex ajustados para capturar apenas o texto da linha (sem o \n no final)
padrao_nome = r'Nome:\s*(.*)'
padrao_codigo = r'Codigo Estacao:\s*(.*)'
padrao_latitude = r'Latitude:\s*(.*)'
padrao_longitude = r'Longitude:\s*(.*)'
padrao_altitude = r'Altitude:\s*(.*)'
padrao_situacao = r'Situacao:\s*(.*)'

# Lista para armazenar os DataFrames de cada arquivo antes do concat (mais eficiente)
lista_dataframes = []

print(f"Iniciando busca por '*.csv' em: {pasta_principal}\n")

for arquivo in pasta_principal.rglob('*.csv'):
    # Evita processar o próprio arquivo de saída caso ele seja gerado na mesma pasta
    if arquivo.name == 'dados.csv':
        continue
        
    try:
        print(f"--- Lendo {arquivo.name} ---")
        
        # 1. Lê os metadados usando re.search (CORREÇÃO DO ERRO 1)
        # Sem re.DOTALL para o regex parar estritamente no fim da linha
        conteudo = arquivo.read_text(encoding='utf-8')
        
        nome_estacao = re.search(padrao_nome, conteudo)
        codigo_estacao = re.search(padrao_codigo, conteudo)
        latitude_estacao = re.search(padrao_latitude, conteudo)
        longitude_estacao = re.search(padrao_longitude, conteudo)
        altitude_estacao = re.search(padrao_altitude, conteudo)
        situacao_estacao = re.search(padrao_situacao, conteudo)

        # Extrai apenas o texto se encontrado, senão define como "Desconhecido"
        nome = nome_estacao.group(1).strip() if nome_estacao else "Desconhecido"
        codigo = codigo_estacao.group(1).strip() if codigo_estacao else "Desconhecido"
        lat = latitude_estacao.group(1).strip() if latitude_estacao else "Desconhecido"
        lon = longitude_estacao.group(1).strip() if longitude_estacao else "Desconhecido"
        alt = altitude_estacao.group(1).strip() if altitude_estacao else "Desconhecido"
        sit = situacao_estacao.group(1).strip() if situacao_estacao else "Desconhecido"

        # 2. Carrega os dados pulando as 10 linhas de metadados
        dados = pd.read_csv(arquivo, sep=';', decimal='.', encoding='utf-8', skiprows=10)
        
        # CORREÇÃO DO ERRO 2: Remove colunas fantasmas geradas pelo ';' no fim da linha
        dados = dados.loc[:, ~dados.columns.str.contains('^Unnamed')]

        # 3. Adiciona as variáveis de metadados de forma limpa (Apenas strings puras)
        dados['nome_estacao'] = nome
        dados['codigo_estacao'] = codigo
        dados['latitude_estacao'] = lat
        dados['longitude_estacao'] = lon
        dados['altitude_estacao'] = alt
        dados['situacao_estacao'] = sit
            
        # Adiciona à lista final
        lista_dataframes.append(dados)
        
    except Exception as e:
        print(f"    Erro ao ler o arquivo {arquivo.name}: {e}")

# 4. Junta todos os arquivos de uma vez só (muito mais rápido do que concatenar um por um no loop)
if lista_dataframes:
    dados_final = pd.concat(lista_dataframes, ignore_index=True)
    
    # Define o caminho final e salva
    caminho_salvamento = pasta_principal / 'dados.csv'
    dados_final.to_csv(caminho_salvamento, index=False, encoding='utf-8-sig')
    print(f"\nProcesso concluído! Arquivo final salvo em: {caminho_salvamento}")
else:
    print("\nNenhum dado foi processado com sucesso.")

Iniciando busca por '*.csv' em: /home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet

--- Lendo dados_83481_D_2009-01-01_2018-01-31.csv ---
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv ---
--- Lendo dados_83692_D_2009-01-01_2026-04-30.csv ---
--- Lendo dados_83587_D_2009-01-01_2026-04-30.csv ---

Processo concluído! Arquivo final salvo em: /home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet/dados.csv
